# PD — XGBoost Challenger

Desarrollo de un modelo **XGBoost Challenger** para estimar la Probability of Default (PD).

El modelo se entrenará utilizando las variables de Feature Engineering construidas bajo un enfoque **leakage-safe**.

El objetivo es comparar un modelo no lineal basado en árboles de boosting contra la **Regresión Logística**, utilizada como modelo interpretable/champion.

### Objetivos

- Entrenar un modelo XGBoost para clasificación binaria.
- Generar probabilidades de default (PD).
- Evaluar AUC-ROC, Gini y KS.
- Comparar el desempeño contra Logistic Regression.
- Analizar si la complejidad adicional de XGBoost aporta una mejora relevante.

### Target

`SeriousDlqin2yrs`

Donde:

- `1` = default / financial distress
- `0` = no default

In [8]:
%pip install xgboost

   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   - -------------------------------------- 2.4/48.9 MB 21.3 MB/s eta 0:00:03
   ---- ----------------------------------- 5.2/48.9 MB 21.9 MB/s eta 0:00:02
   ---------- ----------------------------- 12.6/48.9 MB 24.8 MB/s eta 0:00:02
   ---------------- ----------------------- 19.9/48.9 MB 28.0 MB/s eta 0:00:02
   ----------------------- ---------------- 29.1/48.9 MB 32.7 MB/s eta 0:00:01
   -------------------------------- ------- 39.6/48.9 MB 36.5 MB/s eta 0:00:01
   --------------------------------- ------ 40.9/48.9 MB 31.4 MB/s eta 0:00:01
   -------------------------------------- - 47.2/48.9 MB 31.3 MB/s eta 0:00:01
   ---------------------------------------- 48.9/48.9 MB 30.0 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [9]:
# ============================================
# Imports
# ============================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve
from xgboost import XGBClassifier

print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


In [10]:
# ============================================
# Cargar dataset
# ============================================

DATA_PATH = "../data/raw/cs-training.csv"

df = pd.read_csv(DATA_PATH)

TARGET = "SeriousDlqin2yrs"

print("Dataset cargado correctamente.")
print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]:,}")
print(f"Default rate: {df[TARGET].mean() * 100:.2f}%")

Dataset cargado correctamente.
Filas: 150,000
Columnas: 12
Default rate: 6.68%


In [11]:
# ============================================
# Train / Test Split
# ============================================

X_raw = df.drop(columns=[TARGET])
y_raw = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X_raw,
    y_raw,
    test_size=0.20,
    random_state=42,
    stratify=y_raw
)

print("Train/Test split creado correctamente.")
print(f"Train: {len(X_train):,}")
print(f"Test:  {len(X_test):,}")
print(f"Default rate Train: {y_train.mean() * 100:.2f}%")
print(f"Default rate Test:  {y_test.mean() * 100:.2f}%")

Train/Test split creado correctamente.
Train: 120,000
Test:  30,000
Default rate Train: 6.68%
Default rate Test:  6.68%


## Feature Engineering Leakage-Safe

Para garantizar una comparación justa con la Regresión Logística, XGBoost utilizará exactamente las mismas 14 variables de Feature Engineering.

Las transformaciones que requieren parámetros estadísticos serán ajustadas exclusivamente sobre Train y posteriormente aplicadas a Train y Test.

De esta manera se evita data leakage y ambos modelos son evaluados sobre la misma información.

In [12]:
# ============================================
# Componentes para Feature Engineering
# ============================================

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

print("Componentes cargados correctamente.")

Componentes cargados correctamente.


In [13]:
# ============================================
# Parámetros de Feature Engineering — TRAIN
# ============================================

monthly_income_median_train = X_train["MonthlyIncome"].median()

dependents_median_train = X_train["NumberOfDependents"].median()

debt_ratio_p99_train = X_train["DebtRatio"].quantile(0.99)

utilization_p99_train = (
    X_train["RevolvingUtilizationOfUnsecuredLines"]
    .quantile(0.99)
)

# Edad válida
age_invalid_train = (
    (X_train["age"] < 18) |
    (X_train["age"] > 100)
)

age_median_train = X_train.loc[
    ~age_invalid_train,
    "age"
].median()

# Income per credit line
monthly_income_train = (
    X_train["MonthlyIncome"]
    .fillna(monthly_income_median_train)
)

credit_lines_train = X_train[
    "NumberOfOpenCreditLinesAndLoans"
]

income_per_credit_line_train = (
    monthly_income_train /
    credit_lines_train.replace(0, np.nan)
)

income_credit_median_train = (
    income_per_credit_line_train.median()
)

income_credit_p99_train = (
    income_per_credit_line_train.quantile(0.99)
)

print("Parámetros calculados exclusivamente con TRAIN:")
print(f"MonthlyIncome median: {monthly_income_median_train:.2f}")
print(f"NumberOfDependents median: {dependents_median_train:.2f}")
print(f"Age median: {age_median_train:.2f}")
print(f"DebtRatio P99: {debt_ratio_p99_train:.2f}")
print(f"Utilization P99: {utilization_p99_train:.4f}")
print(f"Income per credit line median: {income_credit_median_train:.2f}")
print(f"Income per credit line P99: {income_credit_p99_train:.2f}")

Parámetros calculados exclusivamente con TRAIN:
MonthlyIncome median: 5390.00
NumberOfDependents median: 0.00
Age median: 52.00
DebtRatio P99: 4977.00
Utilization P99: 1.0944
Income per credit line median: 722.22
Income per credit line P99: 5390.00


In [14]:
# ============================================
# Feature Engineering — Leakage-Safe
# ============================================

def build_pd_features_safe(
    df,
    monthly_income_median,
    dependents_median,
    age_median,
    debt_ratio_p99,
    utilization_p99,
    income_credit_median,
    income_credit_p99
):
    
    data = df.copy()

    # ----------------------------
    # Imputación
    # ----------------------------
    data["MonthlyIncome_imputed"] = (
        data["MonthlyIncome"]
        .fillna(monthly_income_median)
    )

    data["NumberOfDependents_imputed"] = (
        data["NumberOfDependents"]
        .fillna(dependents_median)
    )

    # ----------------------------
    # Edad
    # ----------------------------
    data["age_invalid"] = (
        (data["age"] < 18) |
        (data["age"] > 100)
    ).astype(int)

    data["age_clean"] = (
        data["age"]
        .where(data["age_invalid"] == 0, age_median)
    )

    # ----------------------------
    # Debt Ratio
    # ----------------------------
    data["DebtRatio_extreme"] = (
        data["DebtRatio"] > debt_ratio_p99
    ).astype(int)

    data["DebtRatio_clean"] = (
        data["DebtRatio"]
        .clip(upper=debt_ratio_p99)
    )

    # ----------------------------
    # Utilization
    # ----------------------------
    data["Utilization_extreme"] = (
        data["RevolvingUtilizationOfUnsecuredLines"]
        > utilization_p99
    ).astype(int)

    data["Utilization_clean"] = (
        data["RevolvingUtilizationOfUnsecuredLines"]
        .clip(upper=utilization_p99)
    )

    # ----------------------------
    # Variables de mora
    # ----------------------------
    late_cols = [
        "NumberOfTime30-59DaysPastDueNotWorse",
        "NumberOfTimes90DaysLate",
        "NumberOfTime60-89DaysPastDueNotWorse"
    ]

    for col in late_cols:
        data[f"{col}_imputed"] = data[col].fillna(0)

    data["total_late_count"] = (
        data["NumberOfTime30-59DaysPastDueNotWorse_imputed"]
        + data["NumberOfTimes90DaysLate_imputed"]
        + data["NumberOfTime60-89DaysPastDueNotWorse_imputed"]
    )

    data["max_late_count"] = data[
        [
            "NumberOfTime30-59DaysPastDueNotWorse_imputed",
            "NumberOfTimes90DaysLate_imputed",
            "NumberOfTime60-89DaysPastDueNotWorse_imputed"
        ]
    ].max(axis=1)

    data["late_90plus"] = (
        data["NumberOfTimes90DaysLate_imputed"] > 0
    ).astype(int)

    data["any_late"] = (
        data["total_late_count"] > 0
    ).astype(int)

    # ----------------------------
    # Severidad de mora
    # ----------------------------
    data["late_severity"] = (
        data["NumberOfTime30-59DaysPastDueNotWorse_imputed"]
        + 2 * data["NumberOfTime60-89DaysPastDueNotWorse_imputed"]
        + 3 * data["NumberOfTimes90DaysLate_imputed"]
    )

    data["late_burden"] = (
        data["total_late_count"]
        * data["late_severity"]
    )

    # ----------------------------
    # Capacidad de crédito
    # ----------------------------
    data["income_per_credit_line"] = (
        data["MonthlyIncome_imputed"]
        / data["NumberOfOpenCreditLinesAndLoans"].replace(0, np.nan)
    )

    data["income_per_credit_line_clean"] = (
        data["income_per_credit_line"]
        .fillna(income_credit_median)
        .clip(upper=income_credit_p99)
    )

    return data

In [15]:
# ============================================
# Aplicar Feature Engineering Leakage-Safe
# ============================================

train_fe = build_pd_features_safe(
    X_train,
    monthly_income_median_train,
    dependents_median_train,
    age_median_train,
    debt_ratio_p99_train,
    utilization_p99_train,
    income_credit_median_train,
    income_credit_p99_train
)

test_fe = build_pd_features_safe(
    X_test,
    monthly_income_median_train,
    dependents_median_train,
    age_median_train,
    debt_ratio_p99_train,
    utilization_p99_train,
    income_credit_median_train,
    income_credit_p99_train
)

print("Feature Engineering Leakage-Safe aplicado.")
print(f"Train: {train_fe.shape[0]:,} filas | {train_fe.shape[1]} columnas")
print(f"Test:  {test_fe.shape[0]:,} filas | {test_fe.shape[1]} columnas")

Feature Engineering Leakage-Safe aplicado.
Train: 120,000 filas | 30 columnas
Test:  30,000 filas | 30 columnas


In [16]:
# ============================================
# Features utilizadas por XGBoost
# ============================================

safe_features = [
    "age_clean",
    "DebtRatio_clean",
    "Utilization_clean",
    "MonthlyIncome_imputed",
    "NumberOfDependents_imputed",
    "NumberOfOpenCreditLinesAndLoans",
    "NumberRealEstateLoansOrLines",
    "total_late_count",
    "max_late_count",
    "late_90plus",
    "any_late",
    "late_severity",
    "late_burden",
    "income_per_credit_line_clean"
]

X_xgb_train = train_fe[safe_features].copy()
X_xgb_test = test_fe[safe_features].copy()

print(f"Features XGBoost: {len(safe_features)}")
print(f"NaN Train: {X_xgb_train.isna().sum().sum()}")
print(f"NaN Test:  {X_xgb_test.isna().sum().sum()}")

Features XGBoost: 14
NaN Train: 0
NaN Test:  0


In [17]:
# ============================================
# XGBoost Challenger
# ============================================

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.80,
    colsample_bytree=0.80,
    min_child_weight=5,
    reg_lambda=1.0,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(
    X_xgb_train,
    y_train
)

print("XGBoost Challenger entrenado correctamente.")

XGBoost Challenger entrenado correctamente.


In [18]:
# ============================================
# Predicciones XGBoost
# ============================================

y_xgb_pred_proba = xgb_model.predict_proba(
    X_xgb_test
)[:, 1]

print("Predicciones XGBoost generadas correctamente.")
print(f"Predicciones: {len(y_xgb_pred_proba):,}")
print(f"PD mínima: {y_xgb_pred_proba.min():.4f}")
print(f"PD máxima: {y_xgb_pred_proba.max():.4f}")
print(f"PD media: {y_xgb_pred_proba.mean():.4f}")

Predicciones XGBoost generadas correctamente.
Predicciones: 30,000
PD mínima: 0.0019
PD máxima: 0.9026
PD media: 0.0671


In [19]:
# ============================================
# Métricas XGBoost
# ============================================

auc_xgb = roc_auc_score(
    y_test,
    y_xgb_pred_proba
)

gini_xgb = 2 * auc_xgb - 1

fpr_xgb, tpr_xgb, thresholds_xgb = roc_curve(
    y_test,
    y_xgb_pred_proba
)

ks_xgb_values = tpr_xgb - fpr_xgb
ks_xgb = np.max(ks_xgb_values)

ks_xgb_idx = np.argmax(ks_xgb_values)
ks_xgb_threshold = thresholds_xgb[ks_xgb_idx]

print(f"AUC-ROC : {auc_xgb:.4f}")
print(f"Gini    : {gini_xgb:.4f}")
print(f"KS      : {ks_xgb:.4f}")
print(f"Threshold KS: {ks_xgb_threshold:.4f}")

AUC-ROC : 0.8700
Gini    : 0.7401
KS      : 0.5835
Threshold KS: 0.0724


In [20]:
# ============================================
# Métricas XGBoost
# ============================================

auc_xgb = roc_auc_score(
    y_test,
    y_xgb_pred_proba
)

gini_xgb = 2 * auc_xgb - 1

fpr_xgb, tpr_xgb, thresholds_xgb = roc_curve(
    y_test,
    y_xgb_pred_proba
)

ks_xgb_values = tpr_xgb - fpr_xgb
ks_xgb = np.max(ks_xgb_values)

ks_xgb_idx = np.argmax(ks_xgb_values)
ks_xgb_threshold = thresholds_xgb[ks_xgb_idx]

print(f"AUC-ROC : {auc_xgb:.4f}")
print(f"Gini    : {gini_xgb:.4f}")
print(f"KS      : {ks_xgb:.4f}")
print(f"Threshold KS: {ks_xgb_threshold:.4f}")

AUC-ROC : 0.8700
Gini    : 0.7401
KS      : 0.5835
Threshold KS: 0.0724


In [21]:
# ============================================
# Champion vs Challenger
# ============================================

champion_challenger = pd.DataFrame({
    "model": [
        "Logistic Regression - Leakage-Safe",
        "XGBoost - Challenger"
    ],
    "auc_roc": [
        0.862847,
        auc_xgb
    ],
    "gini": [
        0.725695,
        gini_xgb
    ],
    "ks": [
        0.568547,
        ks_xgb
    ]
})

display(champion_challenger)

,model,auc_roc,gini,ks
0,Logistic Regression - Leakage-Safe,0.862847,0.725695,0.568547
1,XGBoost - Challenger,0.870032,0.740064,0.583540


## Cierre — XGBoost Challenger

En este notebook se desarrolló un modelo **XGBoost Challenger** utilizando las mismas variables de Feature Engineering Leakage-Safe empleadas en la Regresión Logística.

### Comparación Champion vs Challenger

| Modelo | AUC-ROC | Gini | KS |
|---|---:|---:|---:|
| Logistic Regression — Leakage-Safe | 0.8628 | 0.7257 | 0.5685 |
| XGBoost — Challenger | **0.8700** | **0.7401** | **0.5835** |

XGBoost presenta una mejora respecto a la Regresión Logística en las tres métricas de discriminación.

Sin embargo, la diferencia debe evaluarse posteriormente considerando **calibración, estabilidad, capacidad de discriminación por bandas de riesgo e interpretabilidad**.

Por esta razón, XGBoost se mantiene como **Challenger**, mientras que la Regresión Logística permanece como modelo **Champion interpretable** hasta completar la etapa de validación.

### Próximos pasos

1. Evaluar la discriminación de ambos modelos por deciles de riesgo.
2. Analizar calibración de las probabilidades de default.
3. Comparar la concentración de defaults por bandas.
4. Evaluar estabilidad mediante PSI.
5. Seleccionar el modelo final considerando desempeño y criterios de riesgo.